# 03B — Classificação do comprador: somente compras públicas
**TCC — Preços e concentração de fornecedores em compras públicas de insumos hospitalares, 2009–2023**

O BPS admite o registro voluntário de compras por instituições públicas e privadas (Ministério da Saúde, 2016,
*Manual de consulta e análise de preços utilizando o BPS*). Como o objeto do estudo são as compras públicas,
esta rotina separa os registros de instituições privadas e estrangeiras, sem alterar a base harmonizada.

**Entrada:** `Base Harmonizacao Campos/BPS_{ano}.parquet` (saída do 03, não é modificada).
**Saída:** `Base Harmonizacao Publica/BPS_{ano}.parquet` — mesmos arquivos e colunas, só com registros públicos;
`_excluidos/` com os registros removidos; `fluxo_classificacao_comprador.csv` com a contagem por ano e motivo.

**Regra (critério da fonte), aplicada nesta ordem:**

| ordem | motivo | regra |
|---|---|---|
| 1 | `estrangeiro` | instituição "Ministerio de Salud" |
| 2 | decisão manual | CNPJ listado em `DECISAO_MANUAL` |
| 3 | `privado_esfera` | campo `esfera` = PRIVADA (o campo existe nos arquivos de 2009–2016) |
| 4 | `privado_cnpj` | CNPJ classificado como PRIVADA em algum ano de 2009–2016, em qualquer ano da série |
| 5 | `publico` | demais registros |

A partir de 2017 o campo esfera não é informado. A manutenção dos demais registros apoia-se na documentação
atual da fonte, que descreve as bases compiladas como registros de estados e municípios e restringe a consulta
por esfera às esferas federal, estadual e municipal.

In [ ]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────
!pip install polars==1.35.2 pyarrow --quiet
import polars as pl
from pathlib import Path
pl.Config.set_tbl_rows(-1); pl.Config.set_tbl_cols(-1)

BASE          = Path(PASTA_DADOS)
PASTA_ORIGEM  = BASE / 'Base Harmonizacao Campos'
PASTA_PUBLICA = BASE / 'Base Harmonizacao Publica'
PASTA_EXCL    = PASTA_PUBLICA / '_excluidos'          # registros excluidos, para auditoria
PASTA_PUBLICA.mkdir(parents=True, exist_ok=True)
PASTA_EXCL.mkdir(parents=True, exist_ok=True)

# ── DECISAO MANUAL ───────────────────────────────────────────────────────
# O Consorcio Intergestores Parana Saude aparece na fonte ora como PRIVADA,
# ora como ESTADUAL. Por reunir gestores publicos de saude, foi mantido entre
# as compras publicas (decisao da autora).
DECISAO_MANUAL = {
    '03273207000128': 'publico',   # Consorcio Intergestores Parana Saude
}
# ─────────────────────────────────────────────────────────────────────────

arquivos = sorted(PASTA_ORIGEM.glob('BPS_*.parquet'))
print(f'Arquivos de origem: {len(arquivos)}')

## 1. CNPJs classificados como PRIVADA pela fonte

In [ ]:
CNPJ = pl.col('cnpj_instituicao').cast(pl.Utf8).str.strip_chars()
ESF  = pl.col('esfera').cast(pl.Utf8).str.strip_chars().str.to_uppercase()

priv = []
for p in arquivos:
    cols = pl.read_parquet(p, n_rows=0).columns
    if 'esfera' not in cols or 'cnpj_instituicao' not in cols:
        continue
    priv.append(pl.scan_parquet(p)
                  .filter((ESF == 'PRIVADA') & CNPJ.is_not_null() & (CNPJ != ''))
                  .select(CNPJ.alias('cnpj')).unique().collect())
CNPJS_PRIVADOS = pl.concat(priv).unique()['cnpj'].to_list()
print(f'CNPJs privados distintos (sem CNPJ vazio): {len(CNPJS_PRIVADOS)}   # esperado: 52')

## 2. Classificação e gravação da base pública

In [ ]:
NOME = pl.col('instituicao').cast(pl.Utf8).str.strip_chars().str.to_uppercase()

def expr_motivo(cols):
    tem_esf  = 'esfera' in cols
    tem_nome = 'instituicao' in cols
    e = pl.when(NOME.str.contains('MINISTERIO DE SALUD') if tem_nome else pl.lit(False)).then(pl.lit('estrangeiro'))
    if DECISAO_MANUAL:
        e = e.when(CNPJ.is_in(list(DECISAO_MANUAL))).then(CNPJ.replace_strict(DECISAO_MANUAL, default=None))
    if tem_esf:
        e = e.when(ESF == 'PRIVADA').then(pl.lit('privado_esfera'))
    e = e.when(CNPJ.is_in(CNPJS_PRIVADOS)).then(pl.lit('privado_cnpj'))
    return e.otherwise(pl.lit('publico')).alias('motivo')

fluxo = []
for p in arquivos:
    df   = pl.read_parquet(p)
    if 'cnpj_instituicao' not in df.columns:
        df = df.with_columns(pl.lit(None, pl.Utf8).alias('cnpj_instituicao'))
    cols = df.columns
    df   = df.with_columns(expr_motivo(cols))
    ano_compra = (pl.col('data_compra').dt.year() if 'data_compra' in cols else pl.lit(None)).alias('ano_compra')
    fluxo.append(df.with_columns(ano_compra)
                   .group_by('ano_compra', 'motivo')
                   .agg(pl.len().alias('registros'),
                        (pl.col('valor_total').sum() if 'valor_total' in cols else pl.lit(None)).alias('valor'))
                   .with_columns(pl.lit(p.name).alias('arquivo')))
    df.filter(pl.col('motivo') == 'publico').drop('motivo').write_parquet(PASTA_PUBLICA / p.name)
    exc = df.filter(pl.col('motivo') != 'publico')
    if exc.height:
        exc.write_parquet(PASTA_EXCL / p.name.replace('.parquet', '_excluidos.parquet'))
    print(f'{p.name}: {df.height:>8,} lidos | {exc.height:>6,} excluídos')
    del df, exc

fluxo = pl.concat(fluxo, how='diagonal')
fluxo.write_csv(PASTA_PUBLICA / 'fluxo_classificacao_comprador.csv')

## 3. Tabela de fluxo (recorte 2009–2023, pelo ano da compra)

In [ ]:
f = fluxo.filter(pl.col('ano_compra').is_between(2009, 2023))
print(f.group_by('ano_compra', 'motivo').agg(pl.col('registros').sum())
       .pivot(on='motivo', index='ano_compra', values='registros').fill_null(0)
       .sort('ano_compra'))

tot = f.group_by('motivo').agg(pl.col('registros').sum(), pl.col('valor').sum()).sort('registros', descending=True)
tot = tot.with_columns((pl.col('registros') / pl.col('registros').sum() * 100).round(2).alias('pct_registros'),
                       (pl.col('valor') / pl.col('valor').sum() * 100).round(2).alias('pct_valor'))
print(tot)
print('\nValores de referencia do TCC (fixos, para comparar com a tabela acima): publico 851.109 | privado_esfera 29.255 | privado_cnpj 3.220 | estrangeiro 71')


## 4. Diagnóstico: compradores sem esfera na fonte com nome não típico de ente público (2017+)
Lista informativa. Esses registros permanecem na base pública pela regra acima; o arquivo
`casos_revisao_manual.csv` documenta quais são e quantos registros representam (cerca de 0,5% da base).


In [ ]:
conhecidos = []
for p in arquivos:
    cols = pl.read_parquet(p, n_rows=0).columns
    if 'esfera' in cols:
        conhecidos.append(pl.scan_parquet(p)
            .filter(ESF.is_in(['FEDERAL','ESTADUAL','MUNICIPAL','PRIVADA']))
            .select(CNPJ.alias('cnpj')).unique().collect())
conhecidos = pl.concat(conhecidos).unique()

publicos_nome = 'MUNICIP|PREFEITURA|FUNDO |FMS|CONS[OÓ]RCIO|SECRETARIA|ESTADO|ESTADUAL|GOVERNO'
cand = []
for p in sorted(PASTA_PUBLICA.glob('BPS_*.parquet')):
    cols = pl.read_parquet(p, n_rows=0).columns
    if not {'data_compra', 'cnpj_instituicao', 'instituicao'} <= set(cols):
        continue
    cand.append(pl.scan_parquet(p)
        .filter(pl.col('data_compra').dt.year() >= 2017)
        .select(CNPJ.alias('cnpj'), pl.col('instituicao').cast(pl.Utf8).str.strip_chars().alias('instituicao'))
        .collect())
cand = (pl.concat(cand)
          .join(conhecidos, on='cnpj', how='anti')
          .filter(~pl.col('instituicao').str.to_uppercase().str.contains(publicos_nome))
          .group_by('cnpj', 'instituicao').agg(pl.len().alias('registros'))
          .sort('registros', descending=True))
cand.write_csv(PASTA_PUBLICA / 'casos_revisao_manual.csv')
print(cand)
print(f'\nTotal de registros em revisão: {cand["registros"].sum():,}')